# Notebook 2 — UCSF RMAC AML Augmentation Pipeline
**Dataset:** UCSF RMAC | **Format:** HDF5 | **Phase:** Arterial | **Label:** AML only

---

## What This Notebook Does

This notebook performs image-level data augmentation on AML-labeled cases from the UCSF RMAC dataset. Three augmented versions are created per case — eroded mask, dilated mask, and flipped image+mask. Augmented files are saved as new HDF5 files with all original metadata preserved. They are treated as independent new images in Notebook 4 (UCSF feature extraction).

**Pipeline position:**
```
[Notebook 1] → [Notebook 2 — YOU ARE HERE] → [Notebook 3] → [Notebook 4]
 KiTS23 Aug     UCSF Augmentation             KiTS23 Extr   UCSF Extr
```

---

## Same Augmentation Rationale as Notebook 1

Image-level augmentation is performed before feature extraction so all features remain internally consistent. Erosion and dilation simulate segmentation variability (mask only). Flip creates a mirror patient (both image and mask). See Notebook 1 for the full rationale and reference paper.

---

## Key Fixes vs. Original Pipeline

### Fix 1 — Real Voxel Spacing Read and Preserved (Critical)

The original pipeline loaded HDF5 image arrays with `sitk.GetImageFromArray()`, which defaults spacing to `(1.0, 1.0, 1.0) mm` because it has no way to infer real scanner spacing from a raw numpy array. The real spacing was stored in the HDF5 file-level attributes but was never read.

After inspecting the HDF5 files, the real spacing is confirmed to be stored here:
```
f.attrs['arterial_pixdim'] → e.g. [0.761719, 0.761719, 5.0]
f.attrs['mask_pixdim']     → e.g. [0.761719, 0.761719, 5.0]
```

This notebook reads all file-level attributes from the original HDF5 and copies them into every augmented output file. Notebook 4 then reads `f.attrs['arterial_pixdim']` and sets it on the SimpleITK image before calling PyRadiomics.

Without this fix, all UCSF shape features (volume, surface area, diameters) are in voxel units instead of mm, and texture features are computed on the wrong voxel grid — not comparable to KiTS23.

### Fix 2 — HDF5 Dataset Key Corrected

The original save function stored augmented data under the key `'image'`. However, the original HDF5 files and Notebook 4 both use the key `'arterial'`. This mismatch caused Notebook 4 to fail silently when reading augmented files. The corrected save function always uses `'arterial'`.

### Fix 3 — Size-Mismatch Cases Logged and Skipped Gracefully

Three cases have mismatched image/mask array dimensions and cannot be augmented. These are now detected automatically by shape comparison rather than a hardcoded list. They are logged clearly and skipped. Their original files remain available for Notebook 4.

---

## Confirmed HDF5 File Structure

```
case.hdf5
  datasets:
    arterial   shape=(512, 512, 30)  dtype=float64  <- CT arterial phase
    mask       shape=(512, 512, 30)  dtype=float64  <- tumor segmentation
    noncon     shape=(512, 512, 30)  dtype=float64  <- not used, copied
    portven    shape=(512, 512, 30)  dtype=float64  <- not used, copied
  attrs:
    arterial_pixdim  [0.761719, 0.761719, 5.0]  <- REAL SCANNER SPACING
    mask_pixdim      [0.761719, 0.761719, 5.0]
    PID              '1a7jTnjpn9'
    pathology        'angiomyolipoma'
    Manufacturer     'GE MEDICAL SYSTEMS Revolution HD'
    Patient Age      '035Y'
    Patient Sex      'F'
```

---

## Output Directory Structure
```
hdf5-Augmentation/
  eroded/case.hdf5   <- eroded mask, original image, all attrs preserved
  dilated/case.hdf5  <- dilated mask, original image, all attrs preserved
  flipped/case.hdf5  <- flipped image+mask, all attrs preserved
```

## 1. Install Dependencies

In [ ]:
!pip install h5py scipy numpy -q

## 2. Imports

In [ ]:
import os
import h5py
import numpy as np
from scipy import ndimage

print("Libraries loaded.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Paths and Configuration

In [ ]:
SOURCE_DIR          = "/content/drive/MyDrive/hdf5/ucsf-rmac-dataset/hdf5_files/"
OUTPUT_DIR          = "/content/drive/MyDrive/hdf5-Augmentation"
PHASE               = "arterial"
PERTURBATION_FACTOR = 0.30

all_files = sorted([f for f in os.listdir(SOURCE_DIR) if f.endswith(".hdf5")])
print(f"Total HDF5 files found: {len(all_files)}")
print(f"First 5: {all_files[:5]}")

## 5. Inspect One File — Confirm Spacing Attribute Exists

In [ ]:
sample = os.path.join(SOURCE_DIR, all_files[0])
with h5py.File(sample, "r") as f:
    print("Dataset keys :", list(f.keys()))
    print("File attributes:")
    for k, v in f.attrs.items():
        print(f"  {k}: {v}")
    spacing_key = f"{PHASE}_pixdim"
    if spacing_key in f.attrs:
        print(f"\nSpacing '{spacing_key}' confirmed: {f.attrs[spacing_key]}")
    else:
        print(f"\nWARNING: spacing key '{spacing_key}' not found — check attribute names.")

## 6. Core Augmentation Functions

In [ ]:
def get_3d_erosion(mask, factor=0.30):
    """
    Iteratively erodes the 3D mask until the target volume reduction is met.
    Applied to mask only. Image array is passed through unchanged.
    """
    original_area = np.sum(mask)
    target_area   = original_area * (1 - factor)
    perturbed     = np.copy(mask)
    struct        = ndimage.generate_binary_structure(3, 1)

    while np.sum(perturbed) > target_area:
        prev_mask = np.copy(perturbed)
        perturbed = ndimage.binary_erosion(perturbed, structure=struct)
        if np.sum(perturbed) == 0:
            print("    Warning: erosion would eliminate mask. Stopping early.")
            return prev_mask
    return perturbed


def get_3d_dilation(mask, factor=0.30):
    """
    Iteratively dilates the 3D mask until the target volume growth is met.
    Applied to mask only. Image array is passed through unchanged.
    """
    original_area = np.sum(mask)
    target_area   = original_area * (1 + factor)
    perturbed     = np.copy(mask)
    struct        = ndimage.generate_binary_structure(3, 1)

    while np.sum(perturbed) < target_area:
        perturbed = ndimage.binary_dilation(perturbed, structure=struct)
    return perturbed


def run_flip_task(image_data, mask_data, axis=0):
    """
    Flips both image and mask along the given axis.
    HDF5 files have no affine matrix — spacing is stored in attrs
    and preserved by the save function below.
    """
    return (
        np.flip(image_data, axis=axis).copy(),
        np.flip(mask_data,  axis=axis).copy()
    )

## 7. Save Function — Attributes Preserved

All original HDF5 file-level attributes (spacing, PID, pathology, manufacturer) are copied into every augmented output file. This is what allows Notebook 4 to correctly read spacing from augmented files.

In [ ]:
def save_hdf5_volume(augmented_image, augmented_mask, output_path,
                     original_attrs, original_file_path, phase="arterial"):
    """
    Saves an augmented image/mask pair as a new HDF5 file.

    - Stores data under the correct phase key (e.g. 'arterial') — not 'image'.
    - Copies all file-level attributes from the original file, including
      arterial_pixdim so Notebook 4 can read real spacing.
    - Copies non-augmented phases (noncon, portven) unchanged.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with h5py.File(output_path, "w") as f_out:
        # Augmented arterial phase and mask
        f_out.create_dataset(phase,  data=augmented_image, compression="gzip")
        f_out.create_dataset("mask", data=augmented_mask,  compression="gzip")

        # Copy other phases unchanged
        with h5py.File(original_file_path, "r") as f_orig:
            for key in f_orig.keys():
                if key not in [phase, "mask"]:
                    f_out.create_dataset(key, data=f_orig[key][:], compression="gzip")

        # Copy ALL original attributes — including spacing
        for key, val in original_attrs.items():
            f_out.attrs[key] = val

    print(f"  Saved: {output_path}")

## 8. Main Pipeline Controller

In [ ]:
def execute_augmentation_pipeline(source_dir, root_output_dir,
                                  phase="arterial", factor=0.30):
    """
    Iterates over all HDF5 files and produces three augmented versions:
      - eroded mask   (image unchanged)
      - dilated mask  (image unchanged)
      - flipped image + mask
    Detects and skips size-mismatch cases automatically.
    Preserves all original metadata in every output file.
    """
    os.makedirs(root_output_dir, exist_ok=True)
    cases   = sorted([f for f in os.listdir(source_dir) if f.endswith(".hdf5")])
    skipped = []

    print(f"Starting augmentation for {len(cases)} HDF5 files...\n")

    for filename in cases:
        file_path = os.path.join(source_dir, filename)
        print(f"Processing: {filename}")

        try:
            with h5py.File(file_path, "r") as f:
                if phase not in f.keys() or "mask" not in f.keys():
                    print(f"  SKIPPED: missing '{phase}' or 'mask' key.")
                    skipped.append((filename, "missing dataset key"))
                    continue

                image_array = f[phase][:]
                mask_array  = f["mask"][:]
                attrs       = dict(f.attrs)   # includes arterial_pixdim

            # Detect size mismatch automatically
            if image_array.shape != mask_array.shape:
                print(f"  SKIPPED: image {image_array.shape} != mask {mask_array.shape}")
                skipped.append((filename, f"shape mismatch"))
                continue

            if np.sum(mask_array) == 0:
                print(f"  SKIPPED: mask is empty.")
                skipped.append((filename, "empty mask"))
                continue

            if f"{phase}_pixdim" not in attrs:
                print(f"  WARNING: spacing attr '{phase}_pixdim' not found in {filename}.")

            # --- Task A: Erosion ---
            print("  [A] Erosion...")
            eroded = get_3d_erosion(mask_array, factor)
            save_hdf5_volume(image_array, eroded,
                             os.path.join(root_output_dir, "eroded", filename),
                             attrs, file_path, phase)

            # --- Task B: Dilation ---
            print("  [B] Dilation...")
            dilated = get_3d_dilation(mask_array, factor)
            save_hdf5_volume(image_array, dilated,
                             os.path.join(root_output_dir, "dilated", filename),
                             attrs, file_path, phase)

            # --- Task C: Flip ---
            print("  [C] Flip...")
            f_img, f_msk = run_flip_task(image_array, mask_array, axis=0)
            save_hdf5_volume(f_img, f_msk,
                             os.path.join(root_output_dir, "flipped", filename),
                             attrs, file_path, phase)

            print(f"  Done: {filename}\n")

        except Exception as e:
            print(f"  ERROR: {e}")
            skipped.append((filename, str(e)))

    print("=" * 50)
    print(f"Augmentation complete.")
    print(f"  Processed : {len(cases) - len(skipped)}")
    print(f"  Skipped   : {len(skipped)}")
    if skipped:
        for s, r in skipped:
            print(f"    {s}: {r}")

## 9. Run

In [ ]:
execute_augmentation_pipeline(
    source_dir      = SOURCE_DIR,
    root_output_dir = OUTPUT_DIR,
    phase           = PHASE,
    factor          = PERTURBATION_FACTOR
)

## 10. Verify Output — Spacing Preserved

In [ ]:
spacing_key  = f"{PHASE}_pixdim"
sample_fname = sorted([f for f in os.listdir(SOURCE_DIR) if f.endswith(".hdf5")])[0]

with h5py.File(os.path.join(SOURCE_DIR, sample_fname), "r") as f:
    orig_spacing = f.attrs.get(spacing_key, "NOT FOUND")
print(f"Original spacing : {orig_spacing}")

for aug in ["eroded", "dilated", "flipped"]:
    aug_path = os.path.join(OUTPUT_DIR, aug, sample_fname)
    if os.path.exists(aug_path):
        with h5py.File(aug_path, "r") as f:
            aug_spacing = f.attrs.get(spacing_key, "NOT FOUND")
            has_phase   = PHASE in f.keys()
            has_mask    = "mask" in f.keys()
        match = np.allclose(orig_spacing, aug_spacing)
        print(f"{aug:8s}: spacing={'MATCH' if match else 'MISMATCH'} {aug_spacing} | "
              f"'{PHASE}'={has_phase} | 'mask'={has_mask}")
    else:
        print(f"{aug}: FILE NOT FOUND")